# 🟢 PaddleOCR LOCAL FAST V4

Corrige la detección del runtime local.

Ya **no depende de que el repo esté montado en `/content/work`**.
Detecta el entorno local por:
- Python 3.12;
- kernel WSL2;
- GPU visible.

Los archivos temporales, la caché de pip y los modelos viven bajo `/root`,
que en tu configuración Docker ya es persistente.

Versiones:
- PaddlePaddle GPU 3.2.0
- CUDA wheel cu126
- PaddleOCR 3.2.0

No usa venv y no reinicia el kernel.

Durante la instalación muestra:
- barra de descarga real;
- MB descargados / total;
- porcentaje;
- velocidad estimada por `tqdm`;
- consola de `pip`;
- heartbeat si `pip` queda silencioso.

In [ ]:
import sys, platform, subprocess, importlib.metadata as md, socket, pathlib

print("🟢 LOCAL FAST V3 · Paddle 3.2.0 / PaddleOCR 3.2.0")
print()

release = platform.release()
platform_text = platform.platform()
host = socket.gethostname()

print("Python:", sys.version)
print("Platform:", platform_text)
print("Kernel:", release)
print("Hostname:", host)
print("HOME:", pathlib.Path.home())
print()

# El runtime Docker local corre sobre WSL2.
is_wsl = ("microsoft" in release.lower()) or ("wsl" in release.lower())
is_py312 = sys.version_info[:2] == (3, 12)

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
gpu_text = gpu.stdout.strip()

print("GPU:", gpu_text or "No detectada")
print()

if not is_py312:
    raise RuntimeError(
        f"Este notebook espera Python 3.12 del runtime local. "
        f"Encontré {sys.version_info.major}.{sys.version_info.minor}."
    )

if not is_wsl:
    print("⚠️ El kernel no parece WSL2.")
    print("Esto podría ser Colab Cloud. Revisá que estés conectado al runtime local.")
    print("No voy a instalar nada automáticamente en esta celda.")
else:
    print("✅ Runtime WSL2 local detectado.")

def ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return "NO INSTALADO"

print()
print("=== PAQUETES ===")
for name in ["paddlepaddle-gpu","paddleocr","paddlex","torch","pillow","numpy"]:
    print(f"{name:20} {ver(name)}")

In [ ]:
import sys, os, pathlib, subprocess, importlib.metadata as md, platform
import threading, queue, time, re
from tqdm.auto import tqdm

PADDLE = "3.2.0"
OCR = "3.2.0"
INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"

release = platform.release().lower()
if "microsoft" not in release and "wsl" not in release:
    raise RuntimeError(
        "ABORTADO: este kernel no parece WSL2/local. "
        "No voy a descargar Paddle por accidente en Colab Cloud."
    )

os.environ["PIP_CACHE_DIR"] = str(pathlib.Path.home()/".cache"/"pip")
os.environ["PADDLE_PDX_CACHE_HOME"] = str(pathlib.Path.home()/".cache"/"paddlex")
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "BOS"

for d in [
    pathlib.Path(os.environ["PIP_CACHE_DIR"]),
    pathlib.Path(os.environ["PADDLE_PDX_CACHE_HOME"]),
]:
    d.mkdir(parents=True, exist_ok=True)

def ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

def fmt_bytes(n):
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    while n >= 1024 and i < len(units) - 1:
        n /= 1024
        i += 1
    return f"{n:.1f} {units[i]}"

def run_pip(cmd, title):
    """
    Ejecuta pip con feedback real:
    - consola en vivo;
    - barra basada en 'Progress CURRENT of TOTAL';
    - MB descargados / total;
    - velocidad;
    - heartbeat si pip queda silencioso.
    """
    print()
    print("=" * 86)
    print(f"📦 {title}")
    print("=" * 86)
    print("$", " ".join(cmd), flush=True)
    print()

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    q = queue.Queue()

    def reader():
        try:
            for line in iter(proc.stdout.readline, ""):
                q.put(line)
        finally:
            q.put(None)

    threading.Thread(target=reader, daemon=True).start()

    progress_re = re.compile(r"Progress\s+(\d+)\s+of\s+(\d+)", re.I)
    current_bar = None
    current_total = None
    last_value = 0
    last_output = time.time()
    reader_done = False
    started = time.time()

    while proc.poll() is None or not reader_done:
        had_output = False

        while True:
            try:
                item = q.get_nowait()
            except queue.Empty:
                break

            if item is None:
                reader_done = True
                break

            had_output = True
            last_output = time.time()
            line = item.rstrip("\r\n")

            m = progress_re.search(line)
            if m:
                current = int(m.group(1))
                total = int(m.group(2))

                # Nuevo archivo o nuevo total => nueva barra.
                if current_bar is None or current_total != total or current < last_value:
                    if current_bar is not None:
                        current_bar.close()

                    current_total = total
                    last_value = 0
                    current_bar = tqdm(
                        total=total,
                        desc="Descarga",
                        unit="B",
                        unit_scale=True,
                        unit_divisor=1024,
                        dynamic_ncols=True,
                        leave=True,
                    )

                delta = max(0, current - last_value)
                if delta:
                    current_bar.update(delta)
                last_value = current

                pct = (current / total * 100) if total else 0
                current_bar.set_postfix_str(
                    f"{fmt_bytes(current)} / {fmt_bytes(total)} · {pct:.1f}%"
                )
            else:
                # Dejamos visible la consola real para resolver errores.
                if line.strip():
                    print(f"[pip] {line}", flush=True)

        # Si pip no emitió nada durante 3 s, demostrar que sigue vivo.
        now = time.time()
        if proc.poll() is None and now - last_output >= 3:
            elapsed = int(now - started)
            if current_bar is not None and current_total:
                pct = (last_value / current_total * 100) if current_total else 0
                print(
                    f"[{elapsed:>4}s] ⏳ sigue descargando/instalando · "
                    f"{fmt_bytes(last_value)} / {fmt_bytes(current_total)} · {pct:.1f}%",
                    flush=True,
                )
            else:
                print(
                    f"[{elapsed:>4}s] ⏳ pip sigue trabajando...",
                    flush=True,
                )
            last_output = now

        time.sleep(0.15)

    # Vaciar el remanente.
    while True:
        try:
            item = q.get_nowait()
        except queue.Empty:
            break

        if item is None:
            break

        line = item.rstrip("\r\n")
        m = progress_re.search(line)

        if m:
            current = int(m.group(1))
            total = int(m.group(2))

            if current_bar is None or current_total != total or current < last_value:
                if current_bar is not None:
                    current_bar.close()
                current_total = total
                last_value = 0
                current_bar = tqdm(
                    total=total,
                    desc="Descarga",
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                    dynamic_ncols=True,
                    leave=True,
                )

            delta = max(0, current - last_value)
            if delta:
                current_bar.update(delta)
            last_value = current
        elif line.strip():
            print(f"[pip] {line}", flush=True)

    rc = proc.wait()

    if current_bar is not None:
        current_bar.close()

    elapsed = time.time() - started

    if rc != 0:
        print()
        print(f"❌ Falló después de {elapsed:.1f} s (código {rc}).")
        raise subprocess.CalledProcessError(rc, cmd)

    print()
    print(f"✅ {title} terminado en {elapsed:.1f} s.")
    print("=" * 86)
    print()

print("Pip cache persistente:", os.environ["PIP_CACHE_DIR"])
print("Model cache persistente:", os.environ["PADDLE_PDX_CACHE_HOME"])
print()

if ver("paddlepaddle-gpu") == PADDLE:
    print("✅ PaddlePaddle GPU 3.2.0 ya instalado.")
else:
    existing = ver("paddlepaddle-gpu")
    if existing:
        raise RuntimeError(
            f"Hay PaddlePaddle GPU {existing} instalado. "
            "No voy a mezclar versiones automáticamente."
        )

    print("⬇️ Primera instalación de Paddle 3.2.0.")
    print("Vas a ver MB descargados / total y porcentaje real.")
    print("La descarga grande debería ocurrir UNA sola vez gracias a la caché persistente.")

    run_pip(
        [
            sys.executable, "-m", "pip", "install",
            "-v",
            "--progress-bar=raw",
            "--retries", "10",
            "--timeout", "180",
            f"paddlepaddle-gpu=={PADDLE}",
            "-i", INDEX,
        ],
        "PaddlePaddle GPU 3.2.0 · CUDA 12.6",
    )

if ver("paddleocr") == OCR:
    print("✅ PaddleOCR 3.2.0 ya instalado.")
else:
    existing = ver("paddleocr")
    if existing:
        raise RuntimeError(
            f"Hay PaddleOCR {existing} instalado. "
            "No voy a mezclar versiones automáticamente."
        )

    run_pip(
        [
            sys.executable, "-m", "pip", "install",
            "-v",
            "--progress-bar=raw",
            "--retries", "10",
            "--timeout", "180",
            f"paddleocr=={OCR}",
        ],
        "PaddleOCR 3.2.0",
    )

print()
print("✅ Instalación lista.")
print("NO reinicies el kernel.")
print("La siguiente celda usa un proceso Python nuevo.")

In [ ]:
import sys, subprocess, pathlib, os, platform

release = platform.release().lower()
if "microsoft" not in release and "wsl" not in release:
    raise RuntimeError("No parece el runtime local WSL2.")

dev_dir = pathlib.Path.home()/".cache"/"paddleocr-dev"
dev_dir.mkdir(parents=True, exist_ok=True)

worker_path = dev_dir/"paddle_smoke_worker_v3.py"
worker_path.write_text('\nimport os, sys, json\nfrom pathlib import Path\n\nos.environ.setdefault("PADDLE_PDX_MODEL_SOURCE", "BOS")\nos.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(Path.home()/".cache"/"paddlex"))\n\nprint("=== WORKER NUEVO ===", flush=True)\nprint("Python:", sys.version, flush=True)\n\nimport paddle\nprint("Paddle:", paddle.__version__, flush=True)\nprint("CUDA:", paddle.is_compiled_with_cuda(), flush=True)\nprint("GPU count:", paddle.device.cuda.device_count(), flush=True)\n\nif not paddle.is_compiled_with_cuda() or paddle.device.cuda.device_count() < 1:\n    raise RuntimeError("Paddle no ve la GPU CUDA.")\n\npaddle.set_device("gpu:0")\ntry:\n    print("GPU:", paddle.device.cuda.get_device_name(), flush=True)\nexcept Exception:\n    print("GPU: gpu:0", flush=True)\n\nimport PIL\nfrom PIL import Image, ImageDraw\nprint("Pillow:", PIL.__version__, flush=True)\n\nfrom paddleocr import PaddleOCR\nimport paddleocr\nprint("PaddleOCR:", getattr(paddleocr, "__version__", "unknown"), flush=True)\n\ndev_dir = Path.home()/".cache"/"paddleocr-dev"\ndev_dir.mkdir(parents=True, exist_ok=True)\n\nimg_path = dev_dir/"paddle_smoke_es.png"\nimg = Image.new("RGB", (1400, 320), "white")\nImageDraw.Draw(img).text(\n    (50, 100),\n    "Histologia epitelio plano simple prueba OCR espanol 12345",\n    fill="black"\n)\nimg.save(img_path)\n\nprint("Imagen:", img_path, flush=True)\nprint("Inicializando OCR...", flush=True)\n\nocr = PaddleOCR(\n    lang="es",\n    device="gpu:0",\n    use_doc_orientation_classify=False,\n    use_doc_unwarping=False,\n    use_textline_orientation=False,\n)\n\nprint("Ejecutando OCR...", flush=True)\nresults = ocr.predict(str(img_path))\n\ntexts = []\nfor res in results:\n    d = getattr(res, "json", res)\n    if callable(d):\n        d = d()\n    if isinstance(d, dict) and "res" in d:\n        d = d["res"]\n    if isinstance(d, dict):\n        texts += [str(x) for x in d.get("rec_texts", [])]\n\nprint("Textos:", json.dumps(texts, ensure_ascii=False), flush=True)\n\nif not texts:\n    raise RuntimeError("No se reconoció texto.")\n\nprint("✅ SMOKE TEST COMPLETO", flush=True)\n', encoding="utf-8")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PADDLE_PDX_MODEL_SOURCE"] = "BOS"
env["PADDLE_PDX_CACHE_HOME"] = str(pathlib.Path.home()/".cache"/"paddlex")

print("Ejecutando worker fresco:")
print(worker_path)
print()

proc = subprocess.Popen(
    [sys.executable, str(worker_path)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in proc.stdout:
    print(line, end="", flush=True)

rc = proc.wait()
if rc:
    raise RuntimeError(f"Worker falló con código {rc}")